# Bikol Speech Validation — Stage 3

Run MMS-LID-256 language classification on pre-segmented WAV files.
This is a GPU-accelerated alternative to running Stage 3 locally.

**Instructions:**
1. Run stages 1-2 locally: `python tinigbicol.py pipeline my_audio/ my_output/ --skip-classify`
2. Zip the WAVs: `zip -r audio.zip my_output/audio/`
3. Upload audio.zip below
4. Runtime → Run All
5. Download manifest.csv + rejected.csv, place in my_output/

In [ ]:
!pip install -q transformers torch soundfile

In [ ]:
from google.colab import files
import zipfile, io

uploaded = files.upload()

for name, data in uploaded.items():
    if name.endswith('.zip'):
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            z.extractall('audio/')
        print(f'Extracted {name} → audio/')
    else:
        print(f'Skipping {name} (not a zip)')

In [ ]:
from pathlib import Path
from transformers import pipeline
import soundfile as sf

KNOWN_PH_LANGS = {'bcl', 'bto', 'ubl', 'tgl', 'ceb', 'ilo', 'hil', 'war'}

print('Loading MMS-LID-256 (3.9 GB, one-time download)...')
model = pipeline('audio-classification', model='facebook/mms-lid-256')
print('Model loaded. GPU:', 'yes' if model.device.type != 'cpu' else 'no (CPU)')

In [ ]:
import csv
from pathlib import Path

MANIFEST_FIELDS = [
    'segment_id', 'wav_path', 'dialect_label', 'confidence',
    'source_name', 'source_type', 'duration_ms',
    'predicted_lang', 'predicted_score'
]
REJECTED_FIELDS = MANIFEST_FIELDS + ['reject_reason']

manifest_rows = []
rejected_rows = []
counters = {}

wavs = sorted(Path('audio').rglob('*.wav'))
print(f'Found {len(wavs)} WAV files')

for wav in wavs:
    try:
        result = model(str(wav))[0]
        lang = result['label']
        score = round(result['score'], 4)
    except Exception as e:
        lang, score = '', 0.0

    is_ph = lang in KNOWN_PH_LANGS
    dur = int(sf.info(str(wav)).duration * 1000)
    label = wav.stem.split('_')[0] if '_' in wav.stem else ''
    counters[label] = counters.get(label, 0) + 1
    seg_id = f'{label}_{counters[label]:05d}' if label else f'{wav.stem}'

    row = {
        'segment_id': seg_id, 'wav_path': f'audio/{wav.name}',
        'dialect_label': label, 'confidence': 'high',
        'source_name': 'colab', 'source_type': 'colab_pipeline',
        'duration_ms': dur, 'predicted_lang': lang, 'predicted_score': score
    }

    if is_ph:
        manifest_rows.append(row)
        print(f'  Keep: {wav.name:30s} → {lang} ({score:.3f})')
    else:
        rejected_rows.append({**row, 'reject_reason': 'not_ph_language'})
        print(f'  Reject: {wav.name:30s} → {lang} ({score:.3f})')

# Write CSVs
if manifest_rows:
    with open('manifest.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=MANIFEST_FIELDS, extrasaction='ignore')
        w.writeheader(); w.writerows(manifest_rows)
    print(f'\nmanifest.csv: {len(manifest_rows)} rows')

if rejected_rows:
    with open('rejected.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=REJECTED_FIELDS, extrasaction='ignore')
        w.writeheader(); w.writerows(rejected_rows)
    print(f'rejected.csv: {len(rejected_rows)} rows')

print('\nDone. Download the CSVs below ↓')

In [ ]:
from google.colab import files
import os

if os.path.exists('manifest.csv'):
    files.download('manifest.csv')
if os.path.exists('rejected.csv'):
    files.download('rejected.csv')